In [3]:
import gradio as gr
import nltk
from nltk.probability import FreqDist
from nltk.corpus import stopwords
import string
import heapq

# လိုအပ်သော NLTK data များအား download ပြုလုပ်ခြင်း
nltk.download('punkt')
nltk.download('stopwords')

# --- Website-Style Custom CSS ---
custom_css = """
body { background-color: #f1f5f9; }
.nav-bar { 
    background-color: #ffffff; 
    padding: 15px 30px; 
    border-bottom: 2px solid #e2e8f0; 
    margin-bottom: 20px;
}
.hero-box {
    text-align: center;
    padding: 50px 20px;
    background: linear-gradient(135deg, #2563eb, #7c3aed);
    color: white;
    border-radius: 15px;
    margin-bottom: 30px;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);
}
.workspace-card {
    background: white;
    padding: 25px;
    border-radius: 12px;
    box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1);
}
.footer {
    text-align: center;
    padding: 30px;
    color: #64748b;
    font-size: 0.9em;
}
"""

def process_corpus(input_text, file_obj, target_word):
    raw_text = ""
    if file_obj is not None:
        try:
            with open(file_obj.name, 'r', encoding='utf-8') as f:
                raw_text = f.read()
        except Exception as e:
            return f"Error: {str(e)}", "", ""
    elif input_text.strip():
        raw_text = input_text
    else:
        return "⚠️ စာသားထည့်ပါ သို့မဟုတ် .txt ဖိုင်တင်ပေးပါ။", "", ""

    tokens = nltk.word_tokenize(raw_text)
    sentences = nltk.sent_tokenize(raw_text)
    
    stop_words = set(stopwords.words('english'))
    words = [w.lower() for w in tokens if w.lower() not in stop_words and w not in string.punctuation]
    word_frequencies = FreqDist(words)
    
    sent_scores = {}
    for sent in sentences:
        for word in nltk.word_tokenize(sent.lower()):
            if word in word_frequencies:
                sent_scores[sent] = sent_scores.get(sent, 0) + word_frequencies[word]
    
    summary_sentences = heapq.nlargest(5, sent_scores, key=sent_scores.get)
    summary_res = " ".join(summary_sentences) if summary_sentences else "အနှစ်ချုပ်ရန် လုံလောက်သော စာသားမရှိပါ။"

    concordance_list = []
    if target_word.strip():
        search_term = target_word.strip().lower()
        for i, token in enumerate(tokens):
            if token.lower() == search_term:
                start = max(0, i - 5)
                end = min(len(tokens), i + 6)
                context = tokens[start:end]
                concordance_list.append("• ... " + " ".join(context) + " ...")
        concordance_res = "\n\n".join(concordance_list[:15]) if concordance_list else "❌ မတွေ့ရှိပါ။"
    else:
        concordance_res = "💡 ရှာဖွေရန် Keyword ရိုက်ထည့်ပါ။"

    fdist = FreqDist(words)
    stats_res = f"### 📊 Word Count Table\n\n| Word | Count |\n| :--- | :--- |\n"
    for word, count in fdist.most_common(15):
        stats_res += f"| {word.upper()} | {count} |\n"

    return summary_res, concordance_res, stats_res

# --- UI Architecture ---
with gr.Blocks(css=custom_css, title="TextIntel AI") as demo:
    
    # Navigation Bar
    with gr.Row(elem_classes="nav-bar"):
        gr.Markdown("## 🌐 **TextIntel AI** | Intelligent Text Analytics")

    with gr.Tabs() as tabs:
        # --- TAB 1: HOME PAGE ---
        with gr.TabItem("🏠 HOME"):
            with gr.Column(elem_classes="hero-box"):
                gr.Markdown("# Transform Raw Text into Smart Insights")
                gr.Markdown("Leveraging Natural Language Processing to summarize and analyze documents instantly.")
            
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### 🔍 Smart Analysis")
                    gr.Markdown("Extract key sentences and analyze word distributions with high accuracy.")
                with gr.Column():
                    gr.Markdown("### ⚡ Fast Processing")
                    gr.Markdown("Upload large .txt files and get summaries in a matter of seconds.")
                with gr.Column():
                    gr.Markdown("### 🛠️ NLP Driven")
                    gr.Markdown("Built on NLTK framework for professional-grade linguistic analysis.")
            
            

        # --- TAB 2: ANALYZER ---
        with gr.TabItem("🚀 ANALYZER"):
            with gr.Column(elem_classes="workspace-card"):
                with gr.Row():
                    with gr.Column(scale=2):
                        gr.Markdown("### 📥 Input Source")
                        text_input = gr.Textbox(label="စာသားထည့်ရန်", lines=10, placeholder="Paste your text here...")
                        file_input = gr.File(label=".txt ဖိုင်တင်ရန်", file_types=[".txt"])
                        target_word = gr.Textbox(label="Keyword Search", placeholder="e.g., intelligence")
                        btn = gr.Button("RUN ANALYTICS ✨", variant="primary")
                    
                    with gr.Column(scale=3):
                        gr.Markdown("### 📈 Results")
                        out_summary = gr.Textbox(label="AI Summary", lines=5, interactive=False)
                        with gr.Row():
                            out_concordance = gr.Textbox(label="Concordance (Context)", lines=10, interactive=False)
                            out_stats = gr.Markdown(label="Word Statistics")

        # --- TAB 3: ABOUT ---
        with gr.TabItem("👤 ABOUT"):
            with gr.Column(elem_classes="workspace-card"):
                gr.Markdown("""
                ## Project Overview
                **AI-Enhanced Text Intelligence Tool** သည် လူသားတို့၏ ဘာသာစကားကို AI နည်းပညာများဖြင့် စိတ်ဖြာလေ့လာရန် ဖန်တီးထားခြင်း ဖြစ်ပါသည်။
                 🛠️ Core AI Technologies
                * **AI Summarization:** စာသားတစ်ခုအတွင်းရှိ အရေးကြီးဆုံး ဝါကျ ၅ ကြောင်းကို Frequency Score များတွက်ချက်၍ အနှစ်ချုပ်ပေးခြင်း။
                * **Tokenization:** စာသားများကို အစိတ်အပိုင်းငယ်များ (tokens) အဖြစ် ခွဲခြမ်းစိတ်ဖြာခြင်း။
                * **Contextual Search:** စကားလုံးတစ်လုံးချင်းစီ၏ အသုံးပြုပုံကို ပတ်ဝန်းကျင်ဝါကျများနှင့်တကွ ရှာဖွေပေးခြင်း။
                * **Stop-word Filtering:** အဓိပ္ပာယ်ဖော်ဆောင်မှုနည်းသော စကားလုံးများကို ဖယ်ထုတ်၍ အဓိကအချက်အလက်များကိုသာ ဦးစားပေးခြင်း။
                ### Developer Information
                * **အမည်:** မသိမ့်နန္ဒာထက် (PaKaPaTa-002127)
                * **အတန်း:** 5CS (Section-A)
                """)

    gr.Markdown("© 2026 TextIntel AI - University of Computer Studies", elem_classes="footer")

    btn.click(
        fn=process_corpus, 
        inputs=[text_input, file_input, target_word], 
        outputs=[out_summary, out_concordance, out_stats]
    )

if __name__ == "__main__":
    demo.launch()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\Users\User\AppData\Local\Temp\ipykernel_13300\1993406523.py:94: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, title="TextIntel AI") as demo:


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [2]:
import gradio as gr
import nltk
from nltk.probability import FreqDist
import string
import heapq
import re

# Download necessary NLTK data
nltk.download('punkt')

def mm_segmenter(text):
    """
    Tokenizes Myanmar text using Regex by grouping Unicode characters.
    Also captures English alphanumeric words.
    """
    # Regex to capture Myanmar Unicode range and English words
    regex_pattern = r'[\u1000-\u1049\u104A-\u104F]+|[a-zA-Z0-9]+'
    tokens = re.findall(regex_pattern, text)
    return tokens

def process_corpus(input_text, file_obj, target_word):
    raw_text = ""
    if file_obj is not None:
        try:
            with open(file_obj.name, 'r', encoding='utf-8') as f:
                raw_text = f.read()
        except Exception as e:
            return f"Error: {str(e)}", "", ""
    elif input_text.strip():
        raw_text = input_text
    else:
        return "Please enter text or upload a .txt file.", "", ""

    # --- 1. Tokenization ---
    tokens = mm_segmenter(raw_text)
    # Split sentences by Myanmar '။' or English period/question mark
    sentences = re.split(r'(?<=[။?!.])\s*', raw_text)
    
    # --- 2. Stop-words Filtering ---
    # Common Myanmar and English stop-words
    mm_stop_words = {"သည်", "၏", "၌", "တွင်", "နှင့်", "က", "ကို", "လည်း", "သော", "ရှိ", "၍", "မှ", "ဖြင့်", "တို့", "ပါ"}
    eng_stop_words = {"is", "the", "a", "an", "and", "or", "in", "on", "at", "to", "for", "with"}
    
    combined_stop_words = mm_stop_words.union(eng_stop_words)
    punctuation = set(string.punctuation).union({"။", "၊", "...", " ", "“", "”"})

    # Filter words
    filtered_words = [w.lower() for w in tokens if w.lower() not in combined_stop_words and w not in punctuation]
    word_frequencies = FreqDist(filtered_words)
    
    # --- 3. AI Summarization (Sentence Scoring) ---
    sent_scores = {}
    for sent in sentences:
        for word in mm_segmenter(sent.lower()):
            if word in word_frequencies:
                if sent not in sent_scores:
                    sent_scores[sent] = word_frequencies[word]
                else:
                    sent_scores[sent] += word_frequencies[word]
    
    # Extract top 5 sentences
    summary_sentences = heapq.nlargest(5, sent_scores, key=sent_scores.get)
    summary_res = " ".join(summary_sentences) if summary_sentences else "Not enough text to summarize."

    # --- 4. Context Search (Concordance) ---
    concordance_list = []
    if target_word.strip():
        search_term = target_word.strip().lower()
        for i, token in enumerate(tokens):
            if token.lower() == search_term:
                start = max(0, i - 4)
                end = min(len(tokens), i + 5)
                context = tokens[start:end]
                concordance_list.append("... " + " ".join(context) + " ...")
        concordance_res = "\n\n".join(concordance_list[:15]) if concordance_list else "Word not found."
    else:
        concordance_res = "Please enter a word to search."

    # --- 5. Word Statistics (Markdown Table) ---
    stats_res = f"**Total Word Count:** {len(tokens)}\n\n"
    stats_res += "| Word | Frequency |\n"
    stats_res += "| :--- | :--- |\n"
    for word, count in word_frequencies.most_common(15):
        stats_res += f"| {word} | {count} |\n"

    return summary_res, concordance_res, stats_res

# --- UI Design ---


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 Myanmar & English Text Intelligence")
    
    with gr.Tabs():
        # --- Tab 1: Text Explorer ---
        with gr.TabItem("TEXT EXPLORER"):
            gr.Markdown("Analyze text using NLP and Summarization algorithms.")
            with gr.Row():
                with gr.Column(scale=1):
                    file_input = gr.File(label="Upload .txt File", file_types=[".txt"])
                    text_input = gr.Textbox(label="Input Text", lines=8, placeholder="Paste your text here...")
                    target_word = gr.Textbox(label="Search Word", placeholder="e.g., နည်းပညာ or intelligence")
                    
                    with gr.Row():
                        btn = gr.Button("Analyze Text", variant="primary")
                        clear_btn = gr.Button("Clear All")
                    
                with gr.Column(scale=1):
                    out_summary = gr.Textbox(label="✨ AI Summary (Top 5 Sentences)", lines=6)
                    out_concordance = gr.Textbox(label="🔍 Contextual Usage (Concordance)", lines=6)
                    out_stats = gr.Markdown()

        # --- Tab 2: About Us ---
        with gr.TabItem("ABOUT US"):
            with gr.Column():
                gr.Markdown("""
                ℹ️ Project Overview
                * *AI-Enhanced Text Intelligence Tool* သည် လူသားတို့၏ ဘာသာစကားကို AI နည်းပညာများဖြင့် စိတ်ဖြာလေ့လာရန် ဖန်တီးထားခြင်း ဖြစ်ပါသည်။

                🛠️ Core AI Technologies
                * **AI Summarization:** စာသားတစ်ခုအတွင်းရှိ အရေးကြီးဆုံး ဝါကျ ၅ ကြောင်းကို Frequency Score များတွက်ချက်၍ အနှစ်ချုပ်ပေးခြင်း။
                * **Tokenization:** စာသားများကို အစိတ်အပိုင်းငယ်များ (tokens) အဖြစ် ခွဲခြမ်းစိတ်ဖြာခြင်း။
                * **Contextual Search:** စကားလုံးတစ်လုံးချင်းစီ၏ အသုံးပြုပုံကို ပတ်ဝန်းကျင်ဝါကျများနှင့်တကွ ရှာဖွေပေးခြင်း။
                * **Stop-word Filtering:** အဓိပ္ပာယ်ဖော်ဆောင်မှုနည်းသော စကားလုံးများကို ဖယ်ထုတ်၍ အဓိကအချက်အလက်များကိုသာ ဦးစားပေးခြင်း။

                👤 Developer Information
                * အမည်: မသိမ့်နန္ဒာထက် (PaKaPaTa-002127)
                * အတန်း: 5CS(Section-A)
                """)

    btn.click(
        fn=process_corpus, 
        inputs=[text_input, file_input, target_word], 
        outputs=[out_summary, out_concordance, out_stats]
    )

    clear_btn.click(
        fn=lambda: (None, "", "", "", "", ""),
        inputs=None,
        outputs=[file_input, text_input, target_word, out_summary, out_concordance, out_stats]
    )

if __name__ == "__main__":
    demo.launch()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
C:\Users\User\AppData\Local\Temp\ipykernel_13300\4133084855.py:91: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
